# Kapitola 7: Tvorba chatovacích aplikácií
## Rýchly štart s Microsoft Foundry Models API

Tento zápisník je upravený z [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), ktorý obsahuje zápisníky pristupujúce k službám [Azure OpenAI](notebook-azure-openai.ipynb).

> **Poznámka:** GitHub Models sa stiahne koncom júla 2026. Tento zápisník teraz cieli na [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), ktorý ponúka rovnaký bezplatný katalóg modelov a skúsenosť s Azure AI Inference SDK.


# Prehľad  
"Veľké jazykové modely sú funkcie, ktoré mapujú text na text. Keď dostanú vstupný reťazec textu, veľký jazykový model sa snaží predpovedať text, ktorý bude nasledovať"(1). Tento "rýchly štart" notebook predstaví používateľom základné koncepty LLM, hlavné požiadavky balíka na začiatok práce s AML, jemný úvod do navrhovania promptov a niekoľko krátkych príkladov rôznych prípadov použitia. 


## Table of Contents  

[Overview](#overview)  
[How to use OpenAI Service](#how-to-use-openai-service)  
[1. Creating your OpenAI Service](#1.-creating-your-openai-service)  
[2. Installation](#2.-installation)    
[3. Credentials](#3.-credentials)  

[Use Cases](#use-cases)    
[1. Summarize Text](#1.-summarize-text)  
[2. Classify Text](#2.-classify-text)  
[3. Generate New Product Names](#3.-generate-new-product-names)  
[4. Fine Tune a Classifier](#4.fine-tune-a-classifier)  

[References](#references)


### Vytvorte svoj prvý prompt  
Toto krátke cvičenie poskytne základný úvod do odosielania promptov modelu v Microsoft Foundry Models pre jednoduchú úlohu "zhrnutie".


**Kroky**:  
1. Nainštalujte knižnicu `azure-ai-inference` do svojho python prostredia, ak ste tak ešte neurobili.  
2. Načítajte štandardné pomocné knižnice a nastavte si svoje prihlasovacie údaje pre Microsoft Foundry Models.  
3. Vyberte model pre svoju úlohu  
4. Vytvorte jednoduchý prompt pre model  
5. Odoslať svoju požiadavku do modelového API!


### 1. Nainštalujte `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Importujte pomocné knižnice a vytvorte poverenia


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Nájsť správny model  
Modely ako GPT-4o a GPT-4o mini dokážu rozumieť a generovať prirodzený jazyk a sú dostupné v katalógu Microsoft Foundry Models spolu s modelmi od Meta, Mistral, Cohere a Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-4o-mini"


## 4. Návrh promptov  

"Kúzlo veľkých jazykových modelov spočíva v tom, že ich trénovaním na minimalizáciu tejto chyby predikcie na obrovskom množstve textu sa modely naučia koncepty užitočné pre tieto predikcie. Napríklad sa naučia koncepty ako"(1):

* ako správne hláskovať
* ako funguje gramatika
* ako parafrázovať
* ako odpovedať na otázky
* ako viesť konverzáciu
* ako písať v mnohých jazykoch
* ako programovať
* atď.

#### Ako ovládať veľký jazykový model  
"Zo všetkých vstupov do veľkého jazykového modelu je jednoznačne najvplyvnejší textový prompt(1).

Veľké jazykové modely možno motivovať na produkciu výstupu niekoľkými spôsobmi:

Inštrukcia: Povedzte modelu, čo chcete
Dokončenie: Podnietite model, aby dokončil začiatok toho, čo chcete
Ukážka: Ukážte modelu, čo chcete, buď:
Niekoľko príkladov v texte promptu
Stovky alebo tisíce príkladov v trénovacej dátovej sade na doladenie"



#### Existujú tri základné pravidlá pre tvorbu promptov:

**Ukáž a povedz.** Jasne ukážte, čo chcete, buď cez inštrukcie, príklady, alebo ich kombináciu. Ak chcete, aby model zoradil zoznam položiek podľa abecedy alebo klasifikoval odsek podľa sentimentu, ukážte mu, že toto chcete.

**Poskytnite kvalitné dáta.** Ak sa pokúšate vybudovať klasifikátor alebo chcete, aby model nasledoval vzor, uistite sa, že máte dostatok príkladov. Dôkladne si prekontrolujte svoje príklady — model je obyčajne dostatočne inteligentný na to, aby prehliadol základné pravopisné chyby a poskytol odpoveď, ale môže predpokladať, že ide o zámer a môže to ovplyvniť odpoveď.

**Skontrolujte svoje nastavenia.** Nastavenia temperature a top_p určujú, ako deterministicky model generuje odpoveď. Ak žiadate odpoveď, kde je len jedna správna, chcete tieto hodnoty nastaviť nižšie. Ak hľadáte rozmanitejšie odpovede, mali by ste ich nastaviť vyššie. Najčastejšou chybou pri týchto nastaveniach je predpoklad, že sú to ovládače „šikovnosti“ alebo „kreativity“.


Zdroj: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. Odoslať!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Opakujte rovnaký hovor, ako sa výsledky porovnávajú?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Zhrnutie textu  
#### Výzva  
Zhrňte text pridaním 'tl;dr:' na koniec textového úseku. Všimnite si, ako model rozumie tomu, ako vykonať niekoľko úloh bez ďalších pokynov. Môžete experimentovať s opisnejšími promptami než tl;dr, aby ste upravili správanie modelu a prispôsobili zhrnutie, ktoré dostanete(3).  

Nedávna práca preukázala výrazné zlepšenia v mnohých NLP úlohách a benchmarkoch prostredníctvom predtrénovania na veľkom korpuse textu, nasledovanom doladeniam na konkrétnej úlohe. Hoci je architektúra zvyčajne nezávislá od úlohy, táto metóda stále vyžaduje doladenie na špecifických úlohách s tisíckami alebo desaťtisíckami príkladov. Naproti tomu ľudia dokážu všeobecne zvládnuť novú jazykovú úlohu s iba niekoľkými príkladmi alebo jednoduchými pokynmi – niečo, čo súčasné NLP systémy stále značne nedokážu. Tu ukazujeme, že škálovanie jazykových modelov výrazne zlepšuje nezávislý výkon na úlohách s malým počtom príkladov, niekedy dokonca dosahuje konkurencieschopnosť s predchádzajúcimi špičkovými metódami doladenia.  



Tl;dr


# Cvičenia pre niekoľko prípadov použitia  
1. Zhrnutie textu  
2. Klasifikácia textu  
3. Generovanie nových názvov produktov


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Klasifikovať text  
#### Výzva  
Klasifikujte položky do kategórií poskytnutých počas inferencie. V nasledujúcom príklade uvádzame ako kategórie, tak text na klasifikáciu v podnete (*playground_reference). 

Dotaz zákazníka: Dobrý deň, nedávno sa mi zlomilo jedno z kláves na klávesnici laptopu a budem potrebovať náhradný diel:

Klasifikovaná kategória:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Vytvorte nové názvy produktov
#### Výzva
Vytvorte názvy produktov z ukážkových slov. Tu do promptu zahrnieme informácie o produkte, pre ktorý chceme vytvoriť názvy. Tiež poskytneme podobný príklad, aby sme ukázali vzor, ktorý očakávame. Hodnotu teploty sme nastavili vysoko, aby sme zvýšili náhodnosť a získali inovatívnejšie odpovede.

Popis produktu: Domáci výrobný stroj na mliečne koktaily
Základné slová: rýchly, zdravý, kompaktný.
Názvy produktov: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Popis produktu: Pár topánok, ktorý sadne na akúkoľvek veľkosť nohy.
Základné slová: prispôsobivý, sadne, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Referencie  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Najlepšie postupy pre dolaďovanie modelov GPT na klasifikáciu textu](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Pre ďalšiu pomoc  
[Tím pre komercializáciu OpenAI](AzureOpenAITeam@microsoft.com) 


# Prispievatelia
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
